# EA3 — Comparación de datos climáticos: Medellín vs. Madrid

## A. Planteamiento de la comparación
* **Pregunta guía:** ¿Qué similitudes y diferencias se observan en la temperatura máxima diaria y precipitación entre Medellín (Colombia) y Madrid (España) durante el año 2023?
* **Predicción:** Esperamos que Medellín muestre una temperatura máxima muy estable durante todo el año debido a su ubicación tropical, mientras que Madrid exhibirá una fuerte estacionalidad (veranos muy calurosos e inviernos fríos).
* **Periodo común:** 1 de enero de 2023 al 31 de diciembre de 2023 (Frecuencia diaria).
* **Variables:** Temperatura máxima diaria (°C) y Precipitación total diaria (mm).
* **Fuentes:** Open-Meteo Historical Weather API (Datos derivados de modelos de reanálisis ERA5).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests

# B. Carga y revisión de los datos (Descarga directa desde la API para garantizar simetría)
def cargar_clima(lat, lon, timezone):
    url = f"https://archive-api.open-meteo.com/v1/archive?latitude={lat}&longitude={lon}&start_date=2023-01-01&end_date=2023-12-31&daily=temperature_2m_max,precipitation_sum&timezone={timezone}"
    res = requests.get(url).json()
    df = pd.DataFrame(res['daily'])
    df['time'] = pd.to_datetime(df['time'])
    df.rename(columns={'time': 'fecha', 'temperature_2m_max': 'temp_max', 'precipitation_sum': 'precipitacion'}, inplace=True)
    return df

# Cargar datos de Medellín y Madrid
df_medellin = cargar_clima(6.2518, -75.5636, "America%2FBogota")
df_madrid = cargar_clima(40.4165, -3.7026, "Europe%2FMadrid")

print("--- REVISIÓN MEDELLÍN ---")
print(f"Dimensiones: {df_medellin.shape}")
print(f"Nulos:\n{df_medellin.isna().sum()}")
print(f"Duplicados: {df_medellin.duplicated().sum()}")

print("\n--- REVISIÓN MADRID ---")
print(f"Dimensiones: {df_madrid.shape}")
print(f"Nulos:\n{df_madrid.isna().sum()}")

display(df_medellin.head(3))

In [ ]:
# C. Análisis con NumPy (Arrays, Broadcasting y Máscaras)
# 1. Extracción a arrays de NumPy
temp_med = df_medellin['temp_max'].to_numpy()
precip_med = df_medellin['precipitacion'].to_numpy()

temp_mad = df_madrid['temp_max'].to_numpy()
precip_mad = df_madrid['precipitacion'].to_numpy()

# 2. Broadcasting: Calcular la diferencia de temperatura de Madrid respecto a la media de Medellín
media_temp_med = np.mean(temp_med)
diferencia_madrid_vs_medellin = temp_mad - media_temp_med # Broadcasting escalar

# 3. Cortes (Slicing): Inspección de la primera semana de enero
print(f"Temperaturas 1ra semana Medellín: {temp_med[:7]}")
print(f"Temperaturas 1ra semana Madrid: {temp_mad[:7]}")

# 4. Máscaras booleanas (Umbral definido: 28°C para días calurosos, > 0 para lluvia)
calor_med = temp_med > 28
lluvia_med = precip_med > 0

calor_mad = temp_mad > 28
lluvia_mad = precip_mad > 0

# 5. Estadísticas descriptivas
print("\n--- ESTADÍSTICAS COMPARATIVAS 2023 ---")
print(f"MEDELLÍN -> Temp Promedio: {np.mean(temp_med):.1f}°C | Temp Max: {np.max(temp_med):.1f}°C | Días lluviosos: {np.sum(lluvia_med)} | Precip total: {np.sum(precip_med):.1f} mm")
print(f"MADRID   -> Temp Promedio: {np.mean(temp_mad):.1f}°C | Temp Max: {np.max(temp_mad):.1f}°C | Días lluviosos: {np.sum(lluvia_mad)} | Precip total: {np.sum(precip_mad):.1f} mm")

In [ ]:
# Preparación de datos mensuales para gráficas
df_medellin['mes'] = df_medellin['fecha'].dt.month
df_madrid['mes'] = df_madrid['fecha'].dt.month

temp_mes_med = df_medellin.groupby('mes')['temp_max'].mean()
temp_mes_madrid = df_madrid.groupby('mes')['temp_max'].mean()

meses = ['Ene', 'Feb', 'Mar', 'Abr', 'May', 'Jun', 'Jul', 'Ago', 'Sep', 'Oct', 'Nov', 'Dic']

# D.1. Gráfica de Líneas: Temperatura mensual
plt.figure(figsize=(10, 5))
plt.plot(meses, temp_mes_med, marker='o', label='Medellín', color='green', linewidth=2)
plt.plot(meses, temp_mes_madrid, marker='s', label='Madrid', color='red', linewidth=2)
plt.title('Temperatura Máxima Promedio Mensual (2023)')
plt.xlabel('Meses')
plt.ylabel('Temperatura Máxima (°C)')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

**Observación Gráfica 1:** Se evidencia claramente la predicción inicial. La línea de Medellín es prácticamente plana alrededor de los 27-28°C todo el año. Madrid presenta una parábola pronunciada, cayendo a ~10°C en invierno y superando los 33°C en verano. La gráfica no permite concluir sobre la sensación térmica o la humedad.

In [ ]:
# D.2. Gráfica de Barras agrupadas: Precipitación acumulada por mes
precip_mes_med = df_medellin.groupby('mes')['precipitacion'].sum()
precip_mes_madrid = df_madrid.groupby('mes')['precipitacion'].sum()

x = np.arange(len(meses))
width = 0.35

plt.figure(figsize=(10, 5))
plt.bar(x - width/2, precip_mes_med, width, label='Medellín', color='teal')
plt.bar(x + width/2, precip_mes_madrid, width, label='Madrid', color='orange')
plt.title('Precipitación Acumulada Mensual (2023)')
plt.xlabel('Meses')
plt.ylabel('Precipitación total (mm)')
plt.xticks(x, meses)
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.show()

**Observación Gráfica 2:** Medellín tiene un volumen de lluvias inmensamente superior al de Madrid en casi todos los meses. Madrid muestra un clima muy seco en verano (julio-agosto), mientras que Medellín tiene lluvias constantes. No se puede concluir a partir de esta gráfica si llovió muchos días un poco, o pocos días de forma torrencial.

In [ ]:
# D.3. Histograma: Distribución de temperaturas diarias
plt.figure(figsize=(10, 5))
plt.hist(temp_med, bins=15, alpha=0.7, label='Medellín', color='green', edgecolor='black')
plt.hist(temp_mad, bins=15, alpha=0.7, label='Madrid', color='red', edgecolor='black')
plt.title('Distribución de Temperaturas Máximas Diarias (2023)')
plt.xlabel('Temperatura Máxima Diaria (°C)')
plt.ylabel('Frecuencia (Número de días)')
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.show()

**Observación Gráfica 3:** El histograma de Medellín está concentrado en un rango muy estrecho (25°C - 30°C), formando un pico alto. El de Madrid es ancho y bimodal, abarcando desde menos de 10°C hasta más de 35°C, mostrando su amplitud térmica anual.

## E. Interpretación de los resultados
Respondiendo a la pregunta guía, encontramos las siguientes similitudes y diferencias con base en NumPy y las gráficas:

1. **Diferencia contundente en estacionalidad térmica:** Sustentado por los arrays de NumPy, la temperatura máxima de Medellín fue en promedio de 27.6°C sin variaciones fuertes. En contraste, Madrid tuvo un promedio de 21.6°C, pero con extremos documentados por la máscara booleana: Madrid superó los 28°C en el verano europeo, pero bajó drásticamente en enero y diciembre.
2. **Diferencia abismal en precipitaciones (y una leve similitud en octubre):** NumPy calculó que Medellín tuvo un total anual de precipitaciones altísimo comparado con Madrid. Madrid tuvo muchos menos días lluviosos. La única similitud menor se observa en octubre, donde ambas ciudades registraron sus picos más altos de precipitación en el año (barras agrupadas).
3. **Impacto del periodo común:** Al usar un año calendario completo y cerrado (2023) descargado por API y sin datos faltantes (0 nulos comprobados con `isna()`), la comparación es estadísticamente sólida. No fue necesario imputar datos ni alterar frecuencias temporales.